# Week 13 — FINAL SUBMISSION

This is the last round. Strategy: defensive — protect wins, try safe micro-improvements where trends support them.

In [1]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')

import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 12 Data & Add Results

In [2]:
inputs, outputs = load_week_data("../week 12/week12_clean_data.npz")
print_data_summary(inputs, outputs, "Week 12 Data")

# Week 12 submitted points
week12_inputs = {
    1: np.array([0.423000, 0.435000]),
    2: np.array([0.900000, 0.500000]),
    3: np.array([0.347863, 0.672420, 0.445172]),
    4: np.array([0.412690, 0.367443, 0.360391, 0.413441]),
    5: np.array([1.000000, 1.000000, 0.500000, 1.000000]),
    6: np.array([0.755469, 0.275580, 0.644099, 0.672228, 0.162762]),
    7: np.array([0.000000, 0.303263, 0.707844, 0.246481, 0.405689, 0.758028]),
    8: np.array([0.107475, 0.120302, 0.130000, 0.207071, 1.000000, 0.099881, 0.180000, 0.998620])
}

# Week 12 outputs
week12_outputs = {
    1: 0.6354248405244494,
    2: 0.27480359450023384,
    3: -0.012138444039839822,
    4: 0.7210514385479141,
    5: 4678.3425,
    6: -0.5132225381214864,
    7: 1.8830227765424805,
    8: 9.798964173904
}

inputs, outputs = combine_with_week_results(inputs, outputs, week12_inputs, week12_outputs)
print_data_summary(inputs, outputs, "After Week 12 Results")

save_week_data(inputs, outputs, "week13_clean_data.npz")


Data loaded from ../week 12/week12_clean_data.npz

Week 12 Data:
  Function 1: 21 points, 2D, best = 0.964835
  Function 2: 21 points, 2D, best = 0.613847
  Function 3: 26 points, 3D, best = -0.004616
  Function 4: 41 points, 4D, best = 0.724314
  Function 5: 31 points, 4D, best = 8662.482500
  Function 6: 31 points, 5D, best = -0.520742
  Function 7: 41 points, 6D, best = 1.878664
  Function 8: 51 points, 8D, best = 9.796264

After Week 12 Results:
  Function 1: 22 points, 2D, best = 0.964835
  Function 2: 22 points, 2D, best = 0.613847
  Function 3: 27 points, 3D, best = -0.004616
  Function 4: 42 points, 4D, best = 0.724314
  Function 5: 32 points, 4D, best = 8662.482500
  Function 6: 32 points, 5D, best = -0.513223
  Function 7: 42 points, 6D, best = 1.883023
  Function 8: 52 points, 8D, best = 9.798964

Data saved to week13_clean_data.npz


## 2. Week 12 Results Analysis

In [3]:
print("=" * 70)
print("WEEK 12 RESULTS ANALYSIS")
print("=" * 70)

best_before = {}
for fid in range(1, 9):
    best_before[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Best Before W12':>15} {'W12 Query':>12} {'New Best':>12} Status")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    prev_best = best_before[fid]
    w12_val = week12_outputs[fid]
    new_best = np.max(outputs[fid])
    status = "NEW BEST" if w12_val >= prev_best else f"miss"
    if w12_val >= prev_best:
        improved += 1
    print(f"F{fid} {prev_best:>15.4f} {w12_val:>12.4f} {new_best:>12.4f}   {status}")

print(f"\nWeek 12 hit rate: {improved}/8")
print("=" * 70)

print("\nKey Week 12 findings:")
print("- F1: dim2+0.010 OVERSHOT again (0.635 vs 0.965). Both dim1 AND dim2 overshoots confirmed. Spike is VERY narrow.")
print("- F3: dim3+0.002 still overshot (-0.012 vs -0.0046). dim3=0.443 is optimum.")
print("- F4: dim1-0.001 near-miss (0.721 vs 0.724). Very close.")
print("- F6: NEW BEST! dim5-0.0001 broke 12-week stuck streak (+0.008). Continue direction.")
print("- F7: 7th consecutive best via dim2-0.003 (+0.004).")
print("- F8: 4th consecutive best via dim3+0.030 (+0.003).")

WEEK 12 RESULTS ANALYSIS

 F Best Before W12    W12 Query     New Best Status
----------------------------------------------------------------------
F1          0.9648       0.6354       0.9648   miss
F2          0.6138       0.2748       0.6138   miss
F3         -0.0046      -0.0121      -0.0046   miss
F4          0.7243       0.7211       0.7243   miss
F5       8662.4825    4678.3425    8662.4825   miss
F6         -0.5207      -0.5132      -0.5132   NEW BEST
F7          1.8787       1.8830       1.8830   NEW BEST
F8          9.7963       9.7990       9.7990   NEW BEST

Week 12 hit rate: 3/8

Key Week 12 findings:
- F1: dim2+0.010 OVERSHOT again (0.635 vs 0.965). Both dim1 AND dim2 overshoots confirmed. Spike is VERY narrow.
- F3: dim3+0.002 still overshot (-0.012 vs -0.0046). dim3=0.443 is optimum.
- F4: dim1-0.001 near-miss (0.721 vs 0.724). Very close.
- F6: NEW BEST! dim5-0.0001 broke 12-week stuck streak (+0.008). Continue direction.
- F7: 7th consecutive best via dim2-0.003 (+0.

## 3. Week 13 Strategy — FINAL SUBMISSION (BOLD)

### Key insight:
Final score = **MAX across all 13 weeks**. This means:
- Resubmits don't help (max is already recorded)
- Any untested point has ZERO downside to max, positive upside
- **Bolder moves are strictly better** — more upside, same floor

### Philosophy: Every submission is a real attempt. Bold where no evidence says otherwise.

| F | Strategy | Logic | Confidence |
|---|----------|-------|------------|
| F1 | **dim2+0.005 → [0.423, 0.430]** | Proven step size (W5, W7 worked). Safe-bold middle between 0.002 and 0.010. | MODERATE-HIGH |
| F2 | **[0.300, 0.700]** — fresh quadrant | 12 weeks stuck. Last chance for second peak in unexplored NW region. | LOTTERY |
| F3 | **dim2+0.005 → [0.348, 0.677, 0.443]** | Only untested direction. Bold step for max upside. | MODERATE |
| F4 | **dim4+0.010 → [0.414, 0.367, 0.360, 0.4234]** | Fresh dim4 single-dim. Smooth landscape tolerates. | MODERATE |
| F5 | **[1, 1, 1, 0.9999]** — tiniest deviation | Non-zero chance of beating 8662 via noise. | LOTTERY |
| F6 | **dim5-0.0002 → 0.162562** | 2x winning step. Max-scoring = free upgrade. | HIGH |
| F7 | **dim2-0.005 → [0.0, 0.298, ...]** | Original proven step. 8th consecutive target. | HIGH |
| F8 | **dim3+0.030 → [0.107, 0.120, 0.160, ...]** | Full step (not halved). Max-scoring protects overshoots. | HIGH |

In [4]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week13_recommendations = {}

def get_best_point(fid):
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# ============================================================
# F1: dim2+0.005 → [0.423, 0.430] from W10 best
# Max-scoring = bolder is free upgrade. +0.005 is proven step.
# W5, W7 +0.005 worked. W10 +0.010 worked from 0.415.
# +0.005 from 0.425 is safe-bold middle.
# ============================================================
best1 = get_best_point(1)  # [0.423, 0.425] → 0.9648
week13_recommendations[1] = np.array([best1[0], best1[1] + 0.005])

# ============================================================
# F2: [0.300, 0.700] — fresh untested quadrant
# 12 weeks explored dim1 mostly in [0.2, 0.9] with dim2 low.
# Peer reports 0.829 peak somewhere. Last chance.
# [0.3, 0.7] = unexplored NW quadrant. Lottery but no downside.
# ============================================================
week13_recommendations[2] = np.array([0.300000, 0.700000])

# ============================================================
# F3: dim2+0.005 → [0.348, 0.677, 0.443] from W10 best
# dim2- failed (W6). dim1 and dim3 bracketed.
# dim2+ is only untested direction. Bold step for max upside.
# ============================================================
best3_idx = None
for i in range(len(outputs[3])):
    if abs(outputs[3][i] - (-0.0046155839830820155)) < 1e-8:
        best3_idx = i
        break
if best3_idx is None:
    best3_idx = np.argmax(outputs[3])
best3 = inputs[3][best3_idx].copy()
week13_recommendations[3] = np.array([best3[0], best3[1] + 0.005, best3[2]])

# ============================================================
# F4: dim4+0.010 → 0.4234 (fresh dim, meaningful step)
# dim4 has NEVER been tested as single-dim from best.
# Landscape smooth (ls~1.6), 0.010 is still tiny (1/160 ls).
# Bold enough to discover trend if dim4 is productive.
# ============================================================
best4 = get_best_point(4)  # [0.413690, 0.367443, 0.360391, 0.413441] → 0.7243
week13_recommendations[4] = np.array([best4[0], best4[1], best4[2], best4[3] + 0.010])

# ============================================================
# F5: [1, 1, 1, 0.9999] — tiniest untested deviation
# W6 at dim4=0.999 (0.1%) gave 8643. 0.9999 (0.01%) untested.
# Very close to corner, low chance of beating 8662 but non-zero.
# ============================================================
week13_recommendations[5] = np.array([1.000000, 1.000000, 1.000000, 0.999900])

# ============================================================
# F6: dim5-0.0002 → 0.162562 (2x winning step)
# W12's dim5-0.0001 broke 12-week stuck (+0.008).
# Bolder step for bigger potential gain.
# ============================================================
best6 = get_best_point(6)  # W12 new best: dim5=0.162762
week13_recommendations[6] = np.array([
    best6[0],
    best6[1],
    best6[2],
    best6[3],
    best6[4] - 0.0002    # dim5 -0.0002 → 0.162562
])

# ============================================================
# F7: dim2-0.005 → 0.298 (original proven step)
# 7 consecutive bests. Original -0.005 step worked (W6→W7).
# Danger zone at 0.271 still 0.027 away.
# Max-scoring = bolder is strictly better.
# ============================================================
best7 = get_best_point(7)  # W12 best: [0.0, 0.303263, ...] → 1.8830
week13_recommendations[7] = np.array([
    best7[0],
    best7[1] - 0.005,    # dim2 -0.005 (0.303→0.298)
    best7[2],
    best7[3],
    best7[4],
    best7[5]
])

# ============================================================
# F8: dim3+0.030 → 0.160 (original full step)
# 4 consecutive dim3 wins. Max-scoring protects if overshoots.
# ============================================================
best8 = get_best_point(8)  # W12 best: dim3=0.130
week13_recommendations[8] = np.array([
    best8[0],
    best8[1],
    best8[2] + 0.030,    # dim3 +0.030 (0.130→0.160)
    best8[3],
    best8[4],
    best8[5],
    best8[6],
    best8[7]
])

# Sanity check
print("Week 13 FINAL — BOLD PLAN (max-across-weeks scoring)")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week13_recommendations[fid]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    best = np.max(y)
    print(f"F{fid}: best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}")
    print(f"  point: {rec}")
print("=" * 80)

Week 13 FINAL — BOLD PLAN (max-across-weeks scoring)
F1: best=0.9648  pred=0.8274±0.0177
  point: [0.423 0.43 ]
F2: best=0.6138  pred=0.3031±0.2335
  point: [0.3 0.7]
F3: best=-0.0046  pred=-0.0029±0.0014
  point: [0.347863 0.67742  0.443172]
F4: best=0.7243  pred=0.7156±0.0324
  point: [0.41369  0.367443 0.360391 0.423441]
F5: best=8662.4825  pred=8654.6346±2.0515
  point: [1.     1.     1.     0.9999]
F6: best=-0.5132  pred=-0.4989±0.0022
  point: [0.755469 0.27558  0.644099 0.672228 0.162562]
F7: best=1.8830  pred=1.8893±0.0009
  point: [0.       0.298263 0.707844 0.246481 0.405689 0.758028]
F8: best=9.7990  pred=9.7974±0.0020
  point: [0.107475 0.120302 0.16     0.207071 1.       0.099881 0.18     0.99862 ]


## 4. Submission Format

In [5]:
print("=" * 70)
print("WEEK 13 FINAL SUBMISSION — COPY THESE VALUES")
print("=" * 70)

for fid in range(1, 9):
    pt = week13_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")

WEEK 13 FINAL SUBMISSION — COPY THESE VALUES
Function 1:	0.423000-0.430000
Function 2:	0.300000-0.700000
Function 3:	0.347863-0.677420-0.443172
Function 4:	0.413690-0.367443-0.360391-0.423441
Function 5:	1.000000-1.000000-1.000000-0.999900
Function 6:	0.755469-0.275580-0.644099-0.672228-0.162562
Function 7:	0.000000-0.298263-0.707844-0.246481-0.405689-0.758028
Function 8:	0.107475-0.120302-0.160000-0.207071-1.000000-0.099881-0.180000-0.998620
